Groundwater | Flow Modeling Track

# Step 3: From Perceptual to Conceptual – MODFLOW Fundamentals

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → **3-MODFLOW** → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** You've defined model objectives (Step 1) and built a perceptual model of the Limmat Valley aquifer (Step 2). The water balance couldn't be closed because Limmat-aquifer exchange has ×100 uncertainty: this is why we need a numerical model. Now we learn how MODFLOW 6 translates that perceptual model into mathematics.

| **Core content** | ~35-45 minutes |
|:--|:--|
| **Optional: CVFD derivation** | +5 minutes |

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** how MODFLOW 6 discretizes aquifer systems using the Control Volume Finite Difference (CVFD) method
2. **Identify** which MODFLOW 6 packages represent different components of the Limmat Valley perceptual model
3. **Describe** the governing equation that MODFLOW solves and its key assumptions
4. **Recognize** the importance of unit consistency and K-averaging in groundwater modeling

---

## Prerequisites

Before starting this notebook, you should have:
- **Completed [0_start_here.ipynb](../0_start_here.ipynb)** – the 10-step modeling framework
- **Completed [1_model_goal.ipynb](1_model_goal.ipynb)** – model objectives and scope
- **Completed [2_perceptual_model.ipynb](2_perceptual_model.ipynb)** – perceptual model of the Limmat Valley (you'll need the flux estimates from there)
- Basic understanding of Darcy's Law and hydraulic head



In [1]:
# Setting up the notebook
import sys

# Add the _SUPPORT/src to the system path
sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

---

## Introduction



In the previous step, we built a **perceptual model**: a descriptive, high-level understanding of the Limmat Valley aquifer system. Now, we translate that understanding into a **conceptual model**: the bridge between our mental picture and a numerical simulation.

This involves making concrete decisions:
- **Which software?** We use MODFLOW 6, the current USGS standard for groundwater modeling
- **How to discretize?** Dividing continuous space into computational cells (including flexible grids)
- **Which processes?** Selecting packages to represent rivers, wells, recharge, etc.



In this course, we focus on MODFLOW 6 for **groundwater flow modeling**. We use **FloPy**, a Python package for building and running MODFLOW models. You'll work with it hands-on starting in Notebook 4.

 <details>
<summary><strong>Why MODFLOW 6?</strong></summary>

> MODFLOW 6 is the latest generation of MODFLOW, released by the USGS in 2017 and now the standard for groundwater modeling:
> - **Integrated modeling:** Groundwater Flow (GWF) with support for transport and energy models in one framework
> - **Flexible grids:** Supports structured grids, Voronoi grids, and fully unstructured meshes
> - **Open-source & free:** Transparent, reproducible, and trusted by regulators worldwide
> - **Modern architecture:** Python-friendly via FloPy, with parallel computing support
> - **Active development:** Regular updates with new capabilities
</details>



---

## 1 - The Governing Equation

Before discussing discretization, it's important to understand what equation MODFLOW actually solves. MODFLOW simulates groundwater flow by solving the 3D groundwater flow equation, which combines Darcy's Law with the principle of mass conservation:

$$\frac{\partial}{\partial x}\left(K_x \frac{\partial h}{\partial x}\right) + \frac{\partial}{\partial y}\left(K_y \frac{\partial h}{\partial y}\right) + \frac{\partial}{\partial z}\left(K_z \frac{\partial h}{\partial z}\right) + W = S_s \frac{\partial h}{\partial t}$$

Where:
- $h$ is hydraulic head (m)
- $K_x, K_y, K_z$ are hydraulic conductivities in each direction (m/s or m/d)
- $W$ represents sources and sinks—recharge, pumping, river leakage (1/s or 1/d — volume of water per unit volume of aquifer per unit time)
- $S_s$ is specific storage (1/m) — the volume of water released from a unit volume of aquifer per unit decline in head, due to compression of water and aquifer skeleton
- $t$ is time (s or d)

For steady-state conditions (no change in storage over time), the right side equals zero:

$$\nabla \cdot (K \nabla h) + W = 0$$

This is the equation we solve for our initial Limmat Valley model.

> **📚 Theory: Unconfined aquifers (like Limmat Valley)**
>
> For unconfined aquifers, the equation becomes nonlinear because transmissivity $T = K \cdot (h - z_{bot})$ depends on head itself. The saturated thickness $(h - z_{bot})$ changes as the water table rises or falls relative to the aquifer bottom elevation $z_{bot}$. Additionally, storage uses specific yield ($S_y$, the drainable porosity, typically 0.1–0.3) rather than specific storage ($S_s$). MODFLOW handles this automatically when you specify the layer as "convertible."
>
> MODFLOW 6 offers a Newton-Raphson formulation that handles this nonlinearity robustly — we'll use it in Notebook 4. It's activated via the `newtonoptions` setting in FloPy.

**Key assumptions built into MODFLOW:**
- Darcy's Law is valid: Flow is laminar (slow, viscous-dominated flow where fluid particles move in parallel layers—valid when Reynolds number Re < 1–10, which is satisfied in most aquifers)
- Fluid density is constant: No saltwater intrusion or high-concentration effects
- REV assumption: The aquifer can be represented at the scale of model cells

> **💡 What is REV?**
>
> Imagine zooming in on individual sand grains versus zooming out to see the entire beach: at the grain scale, properties vary wildly from solid to void, but at the beach scale, you can meaningfully talk about "average porosity."
>
> REV (Representative Elementary Volume) is the minimum volume over which averaged properties (like porosity and hydraulic conductivity) become meaningful and stable. Below this scale, the porous medium assumption breaks down. See the [REV demo notebook](../../THEORY/_demos/explore_porosity_and_REV.ipynb) for an interactive exploration.

✏️ **Exercise:** Test your understanding of steady state.

In [9]:
from shared_functions import create_multiple_choice

create_multiple_choice("task03_steady_state")


## Steady-State Comprehension
What does **steady state** mean for the groundwater flow equation?


RadioButtons(layout=Layout(width='100%'), options=('A) No water flows through the aquifer', 'B) dh/dt = 0 but …

Output()

Output()

### 1.1 - Optional: How MODFLOW Discretizes the Governing Equation

<details>
<summary><strong> Click to show the optionnal content </strong></summary>

#### From Continuous Equation to Discrete Cells

MODFLOW 6 uses the **Control Volume Finite Difference (CVFD)** method. The idea is simple:

1. Divide the aquifer into cells (control volumes)
2. Apply mass balance to each cell: *flow in − flow out = change in storage*
3. Calculate flow between cells using Darcy's Law

This approach works for any cell shape—rectangles, triangles, or Voronoi polygons.

#### Flow Between Two Connected Cells

Consider two adjacent cells, $i$ and $j$, sharing a face:

```
      Cell i                 Cell j
    +---------+           +---------+
    |         |           |         |
    |   h_i  ----→ Q_ij --→   h_j   |
    |    ·    |    A_ij   |    ·    |
    +---------+           +---------+
              |← L_ij  →|
```

Darcy's Law gives the flow from cell $i$ to cell $j$:

$$Q_{ij} = K \cdot A_{ij} \cdot \frac{h_i - h_j}{L_{ij}}$$

Where:
- $K$ = hydraulic conductivity (m/d)
- $A_{ij}$ = area of the shared face (m²)
- $L_{ij}$ = distance between cell centers (m)
- $h_i, h_j$ = hydraulic heads in each cell (m)

We define **conductance** as the geometric term:

$$C_{ij} = K \cdot \frac{A_{ij}}{L_{ij}} \quad \text{(m²/d)}$$

So flow simplifies to:

$$Q_{ij} = C_{ij} \cdot (h_i - h_j)$$

This is why conductance appears everywhere in MODFLOW!**.
River conductance, GHB conductance, drain conductance—they all follow this pattern: flow = conductance × head difference.

#### Mass Balance at Each Cell

For steady state, the sum of all flows into cell $i$ must equal zero:

$$\sum_{j \in \text{neighbors}} C_{ij}(h_j - h_i) + Q_i^{\text{external}} = 0$$

Where $Q_i^{\text{external}}$ includes wells, recharge, and river leakage.

#### The System of Equations

Writing this mass balance for every cell gives a system of linear equations:

$$\mathbf{A} \cdot \mathbf{h} = \mathbf{b}$$

Where:
- $\mathbf{A}$ = matrix of conductances between cells
- $\mathbf{h}$ = vector of unknown heads (one per cell)
- $\mathbf{b}$ = vector of external sources/sinks

**This is what the IMS solver package solves!** For a model with 10,000 cells, you have 10,000 equations with 10,000 unknowns.

#### Why This Works for Any Grid Shape

The CVFD method only requires:
- Cell centers where head is calculated
- Connections between neighboring cells with defined area and distance

For **Voronoi grids**, this is elegant: cell centers are the generating points, shared edges are perpendicular to the line connecting centers, and distances are well-defined. This is why MODFLOW 6 can handle flexible grids so naturally.

</details>

### 1.2 - Effective K Between Neighboring Cells

When two neighboring cells have **different hydraulic conductivities**, MODFLOW must compute an effective $K_{eff}$ for the connection between them. This matters most when the K contrast is large — for example, a low-K riverbed cell next to a high-K aquifer cell:

```
      Cell i                 Cell j
    +---------+           +---------+
    | K_i     |           |     K_j |
    |  = 100 ----→ Q_ij --→  = 10   |  (m/d)
    |   m/d   |           |   m/d   |
    +---------+           +---------+
              |← L_ij  →|
```

MODFLOW 6 **NPF package** computes $K_{eff}$ using one of several averaging methods. For equal-sized cells:

| Method | Formula | Physical Analogy |
|--------|---------|------------------|
| **Harmonic mean** (MODFLOW default) | $K_{eff} = \frac{2 K_i K_j}{K_i + K_j}$ | Resistors in **series** — low K dominates |
| **Geometric mean** | $K_{eff} = \sqrt{K_i \cdot K_j}$ | Middle ground |
| **Logarithmic mean** | $K_{eff} = \frac{K_i - K_j}{\ln(K_i / K_j)}$ | Between harmonic and arithmetic |
| **Arithmetic mean** | $K_{eff} = \frac{K_i + K_j}{2}$ | Resistors in **parallel** — high K dominates |

The ordering **harmonic $\leq$ geometric $\leq$ logarithmic $\leq$ arithmetic** always holds.

**Why this matters for the Limmat Valley:** When calibrating the model (Notebook 5), you may assign different K values to different zones. At zone boundaries, the averaging method determines how flow transitions between zones. The harmonic mean (MODFLOW's default) is conservative — the low-K zone controls the flow, just like the narrowest pipe segment limits total flow.


✏️ **Exercise:** Use the interactive widget below to explore how these methods diverge as K contrast increases and apply what you learned about K-averaging.

In [10]:
from k_averaging_demo import show_k_averaging
show_k_averaging()

In [11]:
from shared_functions import create_multiple_choice
create_multiple_choice("task03_k_averaging_1")


## K Averaging Checkpoint
Consider flow through two layers in series: a low-K layer (K = 0.1 m/d) overlying a high-K layer (K = 30 m/d). Which K-averaging method best represents the effective conductivity for flow perpendicular to the layers?


RadioButtons(layout=Layout(width='100%'), options=('A) Arithmetic mean — gives equal weight to both K values',…

Output()

Output()

> ⚠️ **Critical: Unit Consistency**
>
> MODFLOW is unit-agnostic: it doesn't enforce any particular unit system. You must ensure all inputs are consistent! Common choices:
>
> | Property | Typical Units | Our Model |
> |----------|--------------|-----------|
> | Length | meters (m) | m |
> | Time | days (d) | d |
> | Hydraulic conductivity | m/d | m/d |
> | Pumping rate | m³/d | m³/d |
> | Recharge | m/d | m/d |
> | Specific storage | 1/m | 1/m |
>
> A common error:* Mixing m/s (from literature) with m/d (from model input) without conversion. 
>
> **Always check units when entering parameter values!**



✏️ **Exercise:** Practice the most common unit conversion in groundwater modeling.

In [12]:
from shared_functions import check_task_with_solution

check_task_with_solution("task03_unit_conversion")


## Unit Conversion Exercise
A laboratory measurement reports hydraulic conductivity as $K = 2.5 \times 10^{-3}$ m/s.
- **Convert this value to m/day for use in your MODFLOW model.**


Output()

Button(description='Show Solution', disabled=True, style=ButtonStyle())

Output()

--

## 2 - Discretizing the World: The MODFLOW Grid

The governing equation cannot be solved analytically for real-world aquifers with complex boundaries and heterogeneous properties. Instead, MODFLOW 6 uses the **Control Volume Finite Difference (CVFD)** method—a numerical technique that divides the domain into cells (control volumes), applies mass balance to each cell, and approximates flow between cells using finite differences. For structured grids this is equivalent to the classical finite-difference method, but CVFD generalizes naturally to any cell shape.

**Traditional structured grids:**
*   **Layers:** Vertical divisions (geological units or computational slices)
*   **Rows & Columns:** Horizontal divisions forming rectangular cells

Each block is called a **cell**, and the model calculates a single hydraulic head value for each cell.

**Flexible grids in MODFLOW 6:**

MODFLOW 6 also supports **unstructured grids** that can adapt to complex geometries:

| Grid Type | Description | Best For |
|-----------|-------------|----------|
| **DIS** | Traditional structured (rows/columns/layers) | Simple domains, beginners |
| **DISV** | Voronoi or quadtree grids (layered) | Local refinement near wells/rivers |
| **DISU** | Fully unstructured | Complex 3D geology |

For our Limmat Valley model, we'll use **DISV (Voronoi grids)** to refine resolution near the rivers while keeping the rest of the domain coarser—balancing accuracy and computational efficiency.

> **What is a Voronoi grid?**
>
> A **Voronoi diagram** (or tessellation) divides space based on proximity to a set of generating points. Each cell contains all locations closer to its generating point than to any other point.
>
> ```
>        A                 B
>     ·------ · ------·
>     |   *   |   *   |      * = generating point
>     | cell  | cell  |
>     |   A   |   B   |      Every location in cell A
>     ·-------·-------·      is closer to point A than
>     |   *   |   *   |      to any other point.
>     | cell  | cell  |
>     |   C   |   D   |
>     ·-------·-------·
>        C                 D
> ```
>
> This makes Voronoi grids ideal for groundwater modeling:
> - Cell centers are the generating points — where MODFLOW calculates head
> - Shared edges are perpendicular to lines connecting cell centers — matching the CVFD method perfectly
> - Easy local refinement — add more generating points where you need detail (wells, rivers)
>
> Use the interactive demo below to compare grid types. Notice how Voronoi and triangular grids concentrate cells near the well location.

In [13]:
from grid_demo import show_grid_comparison

show_grid_comparison()

In [14]:
create_multiple_choice("task03_grid_choice")


## Grid Type Selection
Which grid type allows local refinement near wells and rivers while keeping the rest of the domain coarser?


RadioButtons(layout=Layout(width='100%'), options=('A) Structured (DIS) — uniform rows and columns', 'B) Voron…

Output()

Output()

---

## 3 - The Building Blocks of MODFLOW 6: Packages

MODFLOW 6 is modular. It is a collection of packages that each handle a specific aspect of the simulation. You combine packages to represent your conceptual model.

### Essential Packages

| Package | Purpose | Limmat Valley Use |
|---------|---------|-------------------|
| **DIS/DISV** | Grid discretization | DISV for Voronoi grid |
| **NPF** | Node Property Flow (hydraulic conductivity) | Aquifer hydraulic conductivity |
| **IC** | Initial Conditions | Starting head distribution |
| **STO** | Storage (Ss, Sy) | For transient simulations |
| **CHD** | Constant Head | Western outflow boundary |
| **WEL** | Wells (pumping/injection) | Hardhof wells, lateral inflow |
| **RIV** | River-aquifer interaction | Sihl and Limmat rivers |
| **RCH** | Areal recharge | Precipitation infiltration |
| **OC** | Output Control | What results to save |
| **IMS** | Iterative Model Solution | Solver (required! Every MODFLOW model needs a solver package to iteratively solve the system of equations. The solver controls convergence criteria and iteration limits. Without it, the model won't run.) |




<details>
<summary> Common Misconceptions </summary> 

| Misconception | Reality |
|---------------|---------|
| "MODFLOW calculates pressure" | MODFLOW calculates **hydraulic head** ($h = z + p/\rho g$), not pressure directly |
| "One layer = one aquifer" | Layers are numerical discretization; you can use multiple layers for one hydrogeologic unit |
| "Steady state means no flow" | Steady state means $\partial h/\partial t = 0$; water still flows, but storage doesn't change |
| "Finer grid = better results" | Finer grids increase computation and can introduce convergence difficulties without proportional accuracy gains |
| "Transmissivity replaces conductivity" | Use $K$ for 3D models; $T = K \cdot b$ is only for 2D horizontal flow |

</details>

### Translating the Limmat Valley Perceptual Model

Now that you understand the packages, here's how we translate our perceptual model from Notebook 2 into MODFLOW 6 components. The flux estimates from Notebook 2 (leakage coefficients, recharge rates, lateral inflows) become direct inputs to these packages.

| Limmat Valley Feature | MODFLOW 6 Package | Notes |
| --------------------- | ----------------- | ----- |
| Aquifer extent & moraine boundaries | **DISV** + **IDOMAIN** | Inactive cells outside aquifer. Each DISV cell has an **IDOMAIN** flag: active (1), inactive (0), or pass-through (−1). We'll use IDOMAIN to mask cells outside the aquifer boundary. |
| Variable aquifer thickness | **DISV** | Layer bottom from GIS data |
| Sihl and Limmat rivers | **RIV** | Conductance from leakage coefficients (Notebook 2) |
| Areal recharge (~10% of precipitation) | **RCH** | ~0.0003 m/d = 110 mm/year |
| Hardhof pumping wells | **WEL** | Negative rates = extraction |
| Hardhof recharge basins | **WEL** | Positive rates = injection |
| Lateral inflow from hills | **WEL** or **GHB** | Estimated flux from Notebook 2 (use WEL when the flux is known; use GHB when you want head-dependent exchange with an external source) |
| Western outflow boundary | **CHD** | Fixed head at ~390 m a.s.l. |
| Aquifer hydraulic conductivity | **NPF** | K ≈ 5–50 m/day (pumping tests) |



**💡 Why not 864 m/d?** Literature values for clean gravels cite K ≈ 10⁻² m/s (= 864 m/d), but field-scale K from pumping tests in the Limmat Valley is typically 8–40 m/day, though the full range in the Limmat Valley alluvial gravels extends to roughly 5–50 m/d. This reflects heterogeneity and partial cementation of the alluvial deposits. We use this field-scale range throughout the course.

**🤔 Reflection:** You now know which MODFLOW packages will represent each flux from Notebook 2 water balance. The river-aquifer exchange that had ×100 uncertainty will be computed by the RIV package based on conductance and head differences: the model will constrain this unknown for us.



### Comprehension Check

**✏️ Exercise:** Why do we use a RIV (River) package for the Limmat and Sihl, rather than a CHD (Constant Head) boundary?

In [ ]:
create_multiple_choice("task03_riv_vs_chd")


## River vs. Constant Head
Why do we use a RIV package for the Limmat and Sihl, rather than a CHD boundary?


RadioButtons(layout=Layout(width='100%'), options=('A) RIV allows head-dependent two-way exchange based on con…

Output()

Output()

---

## Summary

In this notebook, you learned:

- **The governing equation** that MODFLOW solves (3D groundwater flow equation based on Darcy's Law and mass conservation)
- **K-averaging** between cells with different properties (harmonic mean as MODFLOW default)
- **Discretization options** including flexible Voronoi grids (DISV) for local refinement
- **How to translate** perceptual model components into MODFLOW 6 packages
- **Critical importance** of unit consistency in all model inputs

---

## What You're Taking Forward

| Concept | You'll Use It In... |
|---------|---------------------|
| DISV (Voronoi) grid | Notebook 4: Building the grid with local refinement |
| Package selection (RIV, WEL, RCH, CHD) | Notebook 4: Assigning boundary conditions |
| Conductance = K × A / L | Notebook 4: River and GHB conductance calculations |
| K-averaging (harmonic mean default) | Notebook 5: Understanding flow at K-zone boundaries |
| Unit consistency (m, d) | Every notebook – always convert to m/day |
| IMS solver | Notebook 4: Running the model |

---

## Documentation Resources

Bookmark these—you'll need them throughout the course:

- **MODFLOW 6:** [modflow6.readthedocs.io](https://modflow6.readthedocs.io/en/stable/)
- **FloPy:** [flopy.readthedocs.io](https://flopy.readthedocs.io/en/stable/)
- **MODFLOW 6 Examples:** [modflow6-examples.readthedocs.io](https://modflow6-examples.readthedocs.io/)

---

## Next Steps

In the next notebook, we translate our conceptual model into a working MODFLOW 6 simulation using FloPy.

**Continue to:** [4_model_implementation.ipynb](4_model_implementation.ipynb) – Building the Limmat Valley Model

**Review if needed:** [2_perceptual_model.ipynb](2_perceptual_model.ipynb) – Perceptual Model